# Verify the preserved task-treatment pilot artifact

Attach exactly one immutable version of `thestonedape/task-aware-eeg2text-task-treatment-pilots`. Set that version number below, enable Internet, and enable the private Kaggle secret `GITHUB_TOKEN`. This CPU-only notebook re-hashes the preserved artifact, cross-checks its frozen decision tables, and never loads model checkpoints or accesses the held-out test.

In [ ]:
REPO_URL = 'https://github.com/thestonedape/task-aware-eeg2text.git'
VERIFIER_COMMIT = '5036e38511e27c8e9aad25985a7a5ed41bb43ef9'
WORKTREE = '/kaggle/working/SemKey'
OUTPUT = '/kaggle/working/task-treatment-pilot-preserved-verification'
DATASET_SLUG = 'thestonedape/task-aware-eeg2text-task-treatment-pilots'
PRESERVED_DATASET_VERSION = None  # Replace with the integer shown in the Kaggle Versions tab.
EXPECTED_MANIFEST_SHA256 = 'a8cc82e59e078b3a05e21c1aeb046ed91070e59e754030b2d63cd4ff1091329a'
EXPECTED_METADATA_SHA256 = 'e59917d7e2e9edbff67b1f51de61acc0bc01015d5fb36986abe9cd7f17f2b3ae'
assert isinstance(PRESERVED_DATASET_VERSION, int) and PRESERVED_DATASET_VERSION >= 1, 'Set the exact immutable Kaggle dataset version number before running'
OUTPUT_PRESERVED_SOURCE_ID = f'kaggle-dataset-thestonedape-task-aware-eeg2text-task-treatment-pilots-version-{PRESERVED_DATASET_VERSION}'
assert len(VERIFIER_COMMIT) == 40
assert all(len(value) == 64 for value in (EXPECTED_MANIFEST_SHA256, EXPECTED_METADATA_SHA256))

In [ ]:
import glob, hashlib, json, os, platform, shutil, subprocess, sys
from kaggle_secrets import UserSecretsClient

def digest(path):
    state = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            state.update(block)
    return state.hexdigest()

github_token = UserSecretsClient().get_secret('GITHUB_TOKEN')
assert github_token, 'Enable the private Kaggle Secret named GITHUB_TOKEN'
askpass = '/kaggle/working/git_askpass.py'
with open(askpass, 'w', encoding='utf-8') as handle:
    handle.write("#!/usr/bin/env python3\nimport os, sys\nprompt = sys.argv[1] if len(sys.argv) > 1 else ''\nprint(os.environ['GITHUB_TOKEN'] if 'Password' in prompt else 'x-access-token')\n")
os.chmod(askpass, 0o700)
clone_env = os.environ.copy()
clone_env.update({'GIT_ASKPASS': askpass, 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': github_token})
if os.path.exists(WORKTREE):
    shutil.rmtree(WORKTREE)
try:
    subprocess.run(['git', 'clone', REPO_URL, WORKTREE], check=True, env=clone_env)
finally:
    os.remove(askpass)
    del github_token, clone_env
subprocess.run(['git', '-C', WORKTREE, 'checkout', '--detach', VERIFIER_COMMIT], check=True)
actual_commit = subprocess.check_output(['git', '-C', WORKTREE, 'rev-parse', 'HEAD'], text=True).strip()
assert actual_commit == VERIFIER_COMMIT
subprocess.run([sys.executable, '-m', 'unittest', 'evaluation.test_verify_task_treatment_pilot_artifact'], check=True, cwd=WORKTREE)
print({'python': platform.python_version(), 'verifier_commit': actual_commit, 'regression': 'PASS'})

In [ ]:
manifest_candidates = glob.glob('/kaggle/input/**/pilot_manifest.json', recursive=True)
artifact_roots = []
required = [
    'run_metadata.json', 'continuation_decision.json',
    'all_seed_metrics.csv', 'frozen_candidate_pools.csv',
    'paired_model_comparisons.csv', 'seed_averaged_predictions.csv',
    'seedwise_contrasts.csv', 'validation_partition.csv',
    'frozen_protocol', 'runs',
]
for path in manifest_candidates:
    root = os.path.dirname(path)
    if all(os.path.exists(os.path.join(root, item)) for item in required):
        artifact_roots.append(root)
artifact_roots = sorted(set(artifact_roots))
assert len(artifact_roots) == 1, ('Attach exactly one complete task-treatment pilot dataset version', artifact_roots, manifest_candidates)
ARTIFACT_ROOT = artifact_roots[0]
assert digest(os.path.join(ARTIFACT_ROOT, 'pilot_manifest.json')) == EXPECTED_MANIFEST_SHA256
assert digest(os.path.join(ARTIFACT_ROOT, 'run_metadata.json')) == EXPECTED_METADATA_SHA256
print({'dataset_slug': DATASET_SLUG, 'dataset_version': PRESERVED_DATASET_VERSION, 'artifact_root': ARTIFACT_ROOT, 'manifest_sha256': EXPECTED_MANIFEST_SHA256})

In [ ]:
if os.path.exists(OUTPUT):
    shutil.rmtree(OUTPUT)
os.makedirs(OUTPUT)
report_path = os.path.join(OUTPUT, 'verification_report.json')
subprocess.run([
    sys.executable,
    os.path.join(WORKTREE, 'evaluation', 'verify_task_treatment_pilot_artifact.py'),
    '--artifact-root', ARTIFACT_ROOT,
    '--output-report', report_path,
    '--output-preserved-source-id', OUTPUT_PRESERVED_SOURCE_ID,
], check=True)
report = json.load(open(report_path, encoding='utf-8'))
assert report['status'] == 'pass'
assert report['output_preserved_source_id'] == OUTPUT_PRESERVED_SOURCE_ID
assert report['pilot_manifest_sha256'] == EXPECTED_MANIFEST_SHA256
assert report['run_metadata_sha256'] == EXPECTED_METADATA_SHA256
assert report['verified_file_count'] == 73
assert report['continuation_selected'] is False
assert report['requirement_counts'] == {'passed': 2, 'failed': 5}
assert report['checks']['all_11_top_level_artifacts_rehashed'] is True
assert report['checks']['all_12_run_summaries_rehashed'] is True
assert report['checks']['all_48_nested_run_artifacts_rehashed'] is True
assert report['checks']['held_out_test_accessed'] is False
assert report['checks']['checkpoints_loaded'] is False
report_sha256 = digest(report_path)
metadata = {
    'status': 'pass',
    'verifier_commit': actual_commit,
    'output_preserved_source_id': OUTPUT_PRESERVED_SOURCE_ID,
    'pilot_manifest_sha256': EXPECTED_MANIFEST_SHA256,
    'run_metadata_sha256': EXPECTED_METADATA_SHA256,
    'verification_report_sha256': report_sha256,
    'continuation_selected': False,
    'held_out_test_accessed': False,
    'checkpoints_loaded': False,
}
metadata_path = os.path.join(OUTPUT, 'verification_run_metadata.json')
with open(metadata_path, 'w', encoding='utf-8') as handle:
    json.dump(metadata, handle, indent=2, sort_keys=True)
    handle.write('\n')
if os.path.exists(WORKTREE):
    shutil.rmtree(WORKTREE)
summary = {
    'status': report['status'],
    'output_preserved_source_id': OUTPUT_PRESERVED_SOURCE_ID,
    'verified_file_count': report['verified_file_count'],
    'requirement_counts': report['requirement_counts'],
    'continuation_selected': report['continuation_selected'],
    'held_out_test_accessed': report['checks']['held_out_test_accessed'],
    'verification_report_sha256': report_sha256,
}
print(summary)
print('TASK-TREATMENT PILOT PRESERVED-ARTIFACT VERIFICATION: PASS')

A terminal PASS verifies preservation and the recorded negative decision. It does **not** turn the pilot into a positive model result and does not authorize richer-factor, retrieval, or generation training. Save the small verification output only after the final PASS marker.